<a href="https://colab.research.google.com/github/Oliwia501/ai4cm-customer-churn-prediction/blob/main/AI4CM_project_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI for Communication and Marketing
# Customer Churn Prediction Project

---

**Dataset:** E-commerce customer profiles, satisfaction scores, and transaction summaries  
**Goal:** Build a predictive model to identify customers at risk of churning and provide actionable marketing insights

---

## Setup & Library Imports

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, auc)

#plots style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (8, 4)

---
# Task A — Data Audit & Quality
## Dataset Overview

In [ ]:
from google.colab import files

df = pd.read_excel('/content/data (2).xlsx', sheet_name='E Comm')

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

In [ ]:
#dtypes and non-null counts
df.info()

In [ ]:
# Basic statistics for numerical columns
df.describe().round(2)

In [ ]:
# Check for duplicates
n_dup = df.duplicated().sum()
print(f"Number of duplicate rows: {n_dup}")

## Missing Values Analysis


In [ ]:
# Count and percentage of missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(missing_df)

In [ ]:
# --- Missingness pattern heatmap ---
missing_cols = [col for col in df.columns if df[col].isnull().any()]
rows_with_missing = df[df[missing_cols].isnull().any(axis=1)][missing_cols]

miss_matrix = rows_with_missing.isnull().astype(int)
sns.heatmap(
    miss_matrix,
    cmap=['#d4e6f1', '#e74c3c'],
    cbar=False,
    xticklabels=True,
    yticklabels=False,
    linewidths=0.0
)
plt.title('Missingness pattern (red = missing, each row = one incomplete record)', fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Imputation Strategy:**

Most columns have a relatively small percentage of missing data (under 10%).
I decided to use **median imputation** for numerical columns because many of them are skewed
(e.g., Tenure, WarehouseToHome, `DaySinceLastOrder). The median is more robust to outliers than the mean.
For HourSpendOnApp and OrderAmountHikeFromlastYear, median also makes sense.

I'm not dropping rows because the missing data seems to be mostly random (MAR), not systematic,
and the dataset is not huge.

In [ ]:
# Impute numerical columns with median
num_cols_with_missing = ['Tenure', 'WarehouseToHome', 'HourSpendOnApp',
                          'OrderAmountHikeFromlastYear', 'CouponUsed',
                          'OrderCount', 'DaySinceLastOrder']

for col in num_cols_with_missing:
    if col in df.columns and df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Imputed '{col}' with median = {median_val:.2f}")

# Drop the 2 rows missing CustomerID and the 2 rows missing
before = len(df)
df.dropna(subset=['CustomerID', 'Gender'], inplace=True)
print(f"Rows dropped: {before - len(df)}")
print(f"Remaining rows: {len(df)}")
print(f"Any missing values left: {df.isnull().sum().sum()}")


## Data Consistency Checks

In [ ]:
# Check ranges for suspicious columns
print("=== WarehouseToHome range ===")
print(df['WarehouseToHome'].describe())

print("\n=== Tenure range ===")
print(df['Tenure'].describe())

print("\n=== DaySinceLastOrder range ===")
print(df['DaySinceLastOrder'].describe())

print("\n=== CashbackAmount range ===")
print(df['CashbackAmount'].describe())

In [ ]:
# WarehouseToHome: values above 100 km seem unrealistic for local delivery
suspicious_wth = df[df['WarehouseToHome'] > 100]
print(f"WarehouseToHome > 100: {len(suspicious_wth)} rows")
print(suspicious_wth[['CustomerID', 'WarehouseToHome']].head(10))

In [ ]:
# Tenure: negative values would be impossible
negative_tenure = df[df['Tenure'] < 0]
print(f"Tenure < 0: {len(negative_tenure)} rows")

# Check unique values in categorical columns
cat_cols = ['PreferredLoginDevice', 'PreferredPaymentMode', 'Gender',
            'PreferedOrderCat', 'MaritalStatus']
for col in cat_cols:
    print(f"\n{col}: {df[col].unique()}")

**What was found:**

- `WarehouseToHome` has some very high values but we'll handle them in the outlier section below.
  They could represent real customers in far-out areas.
- `Tenure` no negatives and the range makes sense for an e-commerce platform.
- Some categorical columns have inconsistencies (e.g., `CC` and `Credit Card` probably mean the same thing, same with `COD` and `Cash on Delivery`). We'll standardize these.
- `PreferredLoginDevice` also has `Phone` and `Mobile Phone` which could be the same.

In [ ]:
# Standardize categorical inconsistencies

# Payment mode: CC → Credit Card, COD → Cash on Delivery
df['PreferredPaymentMode'] = df['PreferredPaymentMode'].replace({
    'CC': 'Credit Card',
    'COD': 'Cash on Delivery'
})

# Login device: Phone → Mobile Phone (treating them the same)
df['PreferredLoginDevice'] = df['PreferredLoginDevice'].replace({
    'Phone': 'Mobile Phone'
})

print("Payment modes after cleaning:")
print(df['PreferredPaymentMode'].value_counts())

print("\nLogin devices after cleaning:")
print(df['PreferredLoginDevice'].value_counts())

## Outlier Analysis


In [ ]:
# Boxplots for key numeric columns
num_cols = ['Tenure', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered',
            'NumberOfAddress', 'OrderAmountHikeFromlastYear', 'CouponUsed',
            'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightsteelblue'))
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xticks([])

plt.suptitle('Boxplots — Numerical Features', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# IQR-based outlier detection
def count_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    return len(outliers), lower, upper

print(f"{'Column':<35} {'Outliers':>10} {'Lower Bound':>15} {'Upper Bound':>15}")
print('-' * 80)
for col in num_cols:
    n_out, lo, hi = count_outliers_iqr(df[col].dropna())
    print(f"{col:<35} {n_out:>10} {lo:>15.2f} {hi:>15.2f}")

**Outlier Decision:**

Looking at the boxplots and IQR output:
- `WarehouseToHome` has some extreme values (e.g. customers living very far). These could be real,
  so I'll cap at the 99th percentile rather than remove them
- `CouponUsed`, `OrderCount`, and `NumberOfAddress` have high-end outliers that likely represent power users, `i'll **keep** these as they are interesting from a marketing perspective.
- `CashbackAmount` shows a right tail but the values aren't unrealistic

In [ ]:
# Cap WarehouseToHome at 99th percentile
cap_wth = df['WarehouseToHome'].quantile(0.99)
df['WarehouseToHome'] = df['WarehouseToHome'].clip(upper=cap_wth)
print(f"WarehouseToHome capped at {cap_wth:.1f} km (99th percentile)")
print(f"New max: {df['WarehouseToHome'].max()}")

## Encoding Categorical Variables

**Strategy:**
- **Label Encoding** for binary or ordinal columns
- **One-Hot Encoding** for nominal columns with more than 2 categories

In [ ]:
# Store a copy of the original for reference
df_original = df.copy()

# Label encode binary / low-cardinality columns
le = LabelEncoder()
label_encode_cols = ['PreferredLoginDevice', 'Gender', 'MaritalStatus', 'PreferedOrderCat']

for col in label_encode_cols:
    df[col] = le.fit_transform(df[col].astype(str))
    print(f"Label encoded: {col}")

# One-hot encode payment mode (multiple categories)
df = pd.get_dummies(df, columns=['PreferredPaymentMode'], drop_first=True, dtype=int)

# Drop CustomerID — not useful for modeling
df.drop(columns=['CustomerID'], inplace=True)

print(f"\nDataset shape after encoding: {df.shape}")
print("\nFeature list:")
print(df.columns.tolist())

### summary
- The dataset had missing values in about 8 columns, mostly numerical. All were imputed with the median.
- I found some categorical inconsistencies (e.g., `CC` vs `Credit Card`) and standardized them.
- Outliers are present in `WarehouseToHome` and we capped them; for other columns I chose to keep extreme values.


---
# Task B — Exploratory Data Analysis (EDA)
## Churn Distribution

how balanced is target variable

In [ ]:
# Churn counts and percentages
churn_counts = df_original['Churn'].value_counts()
churn_pct = df_original['Churn'].value_counts(normalize=True) * 100

print("Churn distribution:")
print(pd.DataFrame({'Count': churn_counts, 'Percentage': churn_pct.round(2)}))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Count plot
axes[0].bar(['Non-Churner (0)', 'Churner (1)'], churn_counts.values,
            color=['steelblue', 'tomato'], edgecolor='white', width=0.5)
axes[0].set_title('Churn Count')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values, labels=['Non-Churner', 'Churner'],
            colors=['steelblue', 'tomato'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Churn Proportion')

plt.suptitle('Target Variable Distribution', fontsize=13)
plt.tight_layout()
plt.show()

**Class Balance:**

The dataset is **skewed towards non-churners** roughly 83% of customers did not churn and about 17% did.


## Satisfaction Score and Churn

How customer satisfaction relates to churn.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Satisfaction score distribution by churn
df_original.groupby('Churn')['SatisfactionScore'].value_counts(normalize=True).unstack().T.plot(
    kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='white'
)
axes[0].set_title('Satisfaction Score by Churn Status')
axes[0].set_xlabel('Satisfaction Score')
axes[0].set_ylabel('Proportion')
axes[0].legend(['Non-Churner', 'Churner'])
axes[0].tick_params(axis='x', rotation=0)

# Complain vs Churn
complain_churn = df_original.groupby(['Complain', 'Churn']).size().unstack()
complain_churn.plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'],
                    edgecolor='white', width=0.5)
axes[1].set_title('Complaints vs Churn')
axes[1].set_xlabel('Has Complaint (0=No, 1=Yes)')
axes[1].set_ylabel('Number of Customers')
axes[1].legend(['Non-Churner', 'Churner'])
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Average satisfaction by churn
print("Average Satisfaction Score:")
print(df_original.groupby('Churn')['SatisfactionScore'].mean().round(3))

print("\nChurn rate by Complain:")
print(df_original.groupby('Complain')['Churn'].mean().round(3))

**Interpretation:**

- Interestingly, **satisfaction score alone doesn't show a strong linear relationship** with churn.
  Churners seem spread across all satisfaction levels, not just the dissatisfied ones.

- **Complaints:** Much clearer signal. Customers who complained have a 31.7% churn rate vs only 10.9% for those who didn't. An unresolved complaint is a far stronger churn trigger than low satisfaction

## Tenure and Churn

How does customer tenure (length of relationship with the platform) relate to churn?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# KDE / distribution of tenure by churn
for churn_val, label, color in [(0, 'Non-Churner', 'steelblue'), (1, 'Churner', 'tomato')]:
    subset = df_original[df_original['Churn'] == churn_val]['Tenure'].dropna()
    axes[0].hist(subset, bins=20, alpha=0.6, label=label, color=color, edgecolor='white', density=True)
axes[0].set_title('Tenure Distribution by Churn Status')
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Boxplot
data_to_plot = [df_original[df_original['Churn']==0]['Tenure'].dropna(),
                df_original[df_original['Churn']==1]['Tenure'].dropna()]
bp = axes[1].boxplot(data_to_plot, labels=['Non-Churner', 'Churner'], patch_artist=True,
                     boxprops=dict(facecolor='lightsteelblue'))
bp['boxes'][1].set_facecolor('lightsalmon')
axes[1].set_title('Tenure Boxplot by Churn Status')
axes[1].set_ylabel('Tenure (months)')

plt.tight_layout()
plt.show()

In [ ]:
print("Average tenure by churn:")
print(df_original.groupby('Churn')['Tenure'].mean().round(2))

**Interpretation:**

There's a clear pattern: **churners tend to have lower tenure** than non-churners.
Customers who just joined the platform are more likely to leave, which makes intuitive sense.
They haven't built loyalty yet and might be testing the service. This is a classic early-churn problem
that retention campaigns should target especially in the first few months.

## DaySinceLastOrder and Churn

How does customer inactivity (days since last order) relate to churn?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for churn_val, label, color in [(0, 'Non-Churner', 'steelblue'), (1, 'Churner', 'tomato')]:
    subset = df_original[df_original['Churn'] == churn_val]['DaySinceLastOrder'].dropna()
    axes[0].hist(subset, bins=20, alpha=0.6, label=label, color=color, edgecolor='white', density=True)
axes[0].set_title('DaySinceLastOrder by Churn Status')
axes[0].set_xlabel('Days Since Last Order')
axes[0].set_ylabel('Density')
axes[0].legend()

# Mean DaySinceLastOrder by churn
means = df_original.groupby('Churn')['DaySinceLastOrder'].mean()
axes[1].bar(['Non-Churner (0)', 'Churner (1)'], means.values,
            color=['steelblue', 'tomato'], edgecolor='white', width=0.5)
axes[1].set_title('Average Days Since Last Order by Churn')
axes[1].set_ylabel('Average Days')
for i, v in enumerate(means.values):
    axes[1].text(i, v + 0.1, f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

**Interpretation:**

Churners don't necessarily have drastically more days since last order.
The distributions overlap a lot. This tells us that **DaySinceLastOrder alone isn't a strong predictor**
of churn, a customer can churn even right after making an order (maybe they're unhappy with it),
and non-churners can go a while without ordering and still come back.
It may become more useful in combination with other features.

## Correlation Analysis

In [ ]:
# Correlation matrix on the encoded dataframe
# only numeric columns
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 7})
ax.set_title('Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with Churn
churn_corr = corr['Churn'].drop('Churn').sort_values(key=abs, ascending=False)
print("Top 10 correlations with Churn:")
print(churn_corr.head(10).round(3))

**Interpretation:**

The strongest correlations with `Churn` come from:
- **Tenure** (negative) — longer-tenured customers churn less
- **Complain** (positive) — customers who complained are more likely to churn
- **CashbackAmount** (negative) — higher cashback is associated with lower churn (loyalty incentive?)
- **SatisfactionScore** has a weaker correlation than expected

###summary - Task B

- The dataset is **imbalanced (~17% churners)**
- **Tenure and Complain** are the most informative features for predicting churn.
- **Satisfaction score alone** is not as predictive as expected.
- **Cashback incentives** seem to reduce churn, which is an actionable insight.

---
# Task C — Modeling & Evaluation
## Feature Selection

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

print(f"Features: {X.shape[1]}")
print(f"Target distribution:\n{y.value_counts()}")

## Train-Test Split

Standard 80/20 split. I use stratify=y to make sure both sets have roughly the same
churn ratio, since dataset is imbalanced.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nTrain churn rate: {y_train.mean():.2%}")
print(f"Test churn rate:  {y_test.mean():.2%}")

## Feature Scaling

Logistic Regression is sensitive to the scale of features,  large-range variables like `CashbackAmount` will dominate smaller ones like `HourSpendOnApp`.  I use `StandardScaler` to normalize everything to mean=0, std=1.

Random Forest doesn't actually need scaling (it's tree-based), but I apply it anyway
for consistency and to avoid potential issues.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # fit only on train, transform both

print("Scaling done — features have mean ~0 and std ~1 in the training set.")

---
## Model 1: Logistic Regression

A simple, interpretable baseline. Good for understanding which features matter most
(via coefficients) and fast to train.

In [ ]:
# Train Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train)

### Model 1 — Predictions & Evaluation

In [ ]:
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print("=== Logistic Regression — Classification Report ===")
print(classification_report(y_test, y_pred_lr, target_names=['Non-Churner', 'Churner']))

In [ ]:
# Confusion Matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Predicted: No', 'Predicted: Yes'],
            yticklabels=['Actual: No', 'Actual: Yes'])
ax.set_title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.show()

---
## Model 2: Random Forest

A non-linear ensemble model that often outperforms Logistic Regression on tabular data.
It also gives us feature importances directly.

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42,
                             class_weight='balanced', n_jobs=-1)
rf.fit(X_train_scaled, y_train)


### Model 2 — Predictions & Evaluation

In [ ]:
y_pred_rf = rf.predict(X_test_scaled)
y_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]

print("=== Random Forest — Classification Report ===")
print(classification_report(y_test, y_pred_rf, target_names=['Non-Churner', 'Churner']))

In [ ]:
# Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=['Predicted: No', 'Predicted: Yes'],
            yticklabels=['Actual: No', 'Actual: Yes'])
ax.set_title('Random Forest — Confusion Matrix')
plt.tight_layout()
plt.show()

---
## Why Recall Matters for Churn Prediction

In churn prediction, we have two types of errors:

| Error Type | What happens |
|---|---|
| **False Negative** | We miss a churner — they leave and we do nothing |
| **False Positive** | We incorrectly flag a loyal customer as at-risk — we contact them unnecessarily |

**Missing a churner (False Negative) is much more expensive** than bothering a loyal customer
with a retention offer. A loyal customer who gets a surprise coupon will just use it happily.
A churner we missed is **lost revenue** — and acquiring a new customer costs 5–7× more than
retaining an existing one.

That's why **Recall** (= how many actual churners we correctly caught) is the priority metric here.
High recall means we're catching most of the people who are about to leave, even if we
flag some extra loyal customers along the way.

---
## Lift Analysis ????

**What is Lift?**

Lift measures how much better our model is compared to randomly contacting customers.

If we randomly target 10% of our customer base, we'd expect to catch 10% of churners.
If our model gives us a **Lift of 3** at the 10% mark, it means we catch **30% of churners**
by targeting just the top 10% — that's 3× more efficient than random.

**Why marketers love Lift:** It directly translates to campaign efficiency and budget savings.
Instead of contacting 100% of customers, we focus resources on those most likely to churn.

In [ ]:
def compute_lift_curve(y_true, y_prob):
    """Returns cumulative lift at each decile."""
    df_lift = pd.DataFrame({'prob': y_prob, 'actual': y_true})
    df_lift = df_lift.sort_values('prob', ascending=False).reset_index(drop=True)
    total_positives = y_true.sum()

    # Calculate cumulative lift at each decile
    deciles = np.arange(0.1, 1.1, 0.1)
    lifts = []
    for d in deciles:
        n = int(len(df_lift) * d)
        caught = df_lift.iloc[:n]['actual'].sum()
        lift = (caught / n) / (total_positives / len(df_lift))
        lifts.append(lift)
    return deciles * 100, lifts

deciles, lift_lr = compute_lift_curve(y_test.values, y_prob_lr)
_, lift_rf = compute_lift_curve(y_test.values, y_prob_rf)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(deciles, lift_lr, marker='o', label='Logistic Regression', color='steelblue')
ax.plot(deciles, lift_rf, marker='s', label='Random Forest', color='tomato')
ax.axhline(y=1, color='gray', linestyle='--', label='Random Baseline (Lift=1)')
ax.set_xlabel('% of Customers Contacted (by model score, high-risk first)')
ax.set_ylabel('Lift')
ax.set_title('Lift Curve — Both Models vs Random Baseline')
ax.legend()
ax.set_xticks(deciles)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print("\nLift at top 10% of customers:")
print(f"  Logistic Regression: {lift_lr[0]:.2f}x")
print(f"  Random Forest:       {lift_rf[0]:.2f}x")

## Final Model Comparison

In [ ]:
# Build comparison table
models_results = {}
for name, y_pred, y_prob in [('Logistic Regression', y_pred_lr, y_prob_lr),
                               ('Random Forest', y_pred_rf, y_prob_rf)]:
    models_results[name] = {
        'Accuracy':  round(accuracy_score(y_test, y_pred), 3),
        'Precision': round(precision_score(y_test, y_pred), 3),
        'Recall':    round(recall_score(y_test, y_pred), 3),
        'F1 Score':  round(f1_score(y_test, y_pred), 3),
        'Lift @10%': round(compute_lift_curve(y_test.values, y_prob)[1][0], 2)
    }

comparison_df = pd.DataFrame(models_results).T
print(comparison_df)

**Which model to recommend?**

Based on the comparison, **Random Forest** is the recommended model:
- It achieves better Recall (catching more actual churners) and F1 Score
- Its Lift at the top 10% is higher, meaning it's more efficient for targeted campaigns
- It handles non-linear relationships between features naturally


---
# Task D — Business Insights & Marketing Recommendations
## Top 3 Churn Drivers

Using Random Forest feature importances to identify what's really driving churn.

In [ ]:
# Feature importances from Random Forest
feat_imp = pd.Series(rf.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.head(15).plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.invert_yaxis()
ax.set_title('Random Forest — Top 15 Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 5 features:")
print(feat_imp.head(5).round(4))

### Churn Driver #1: Tenure

Customers with shorter tenure are much more likely to churn.

- **Business interpretation:** New customers haven't built loyalty yet. They're still evaluating whether
the platform is worth staying on. The first few months are critical — if they don't have a good
early experience, they leave.

- **Recommendation:** Launch an **onboarding retention program** for customers in their first 3 months: welcome emails, first-purchase cashback, personal account manager follow-up.

---

### Churn Driver #2: Complaints

Customers who raised a complaint in the last month have a significantly higher churn probability.

- **Business interpretation:** Complaints are a warning signal. A customer who complains is already
frustrated,  if the issue isn't resolved quickly and satisfactorily, they will leave.

- **Recommendation:** Build a **complaint fast-response protocol**: flag all complainers for priority customer service follow-up within 24h. Offer a resolution + a goodwill coupon.

---

### Churn Driver #3: Cashback Amount

Customers receiving lower cashback amounts tend to churn more.

- **Business interpretation:** Cashback is perceived as a reward for loyalty. When customers feel
they're not getting value back, they start looking elsewhere.

- **Recommendation:** Create a **personalized cashback booster** for at-risk segments.
Instead of flat-rate cashback for everyone, increase cashback % for customers who show early churn signals (low tenure + recent complaint).

## Marketing Strategy

Based on the model output and churn drivers, here's a proposed action plan:

| Customer Risk Group | Communication Channel | Message Tone | Campaign Idea |
|---|---|---|---|
| **High Risk** (Tenure < 3m + Complaint) | SMS + Email | Urgent, caring | "We noticed an issue — here's 15% cashback on your next order as an apology" |
| **High Risk** (Low Tenure, No Complaint) | Push Notification + Email | Welcoming, friendly | "You've been with us for [X] days — here's a loyalty reward just for you" |
| **Medium Risk** (Mid Tenure, Low Cashback) | Email | Value-focused | "Exclusive member cashback offer — limited time" |
| **Low Risk** (High Tenure, High Satisfaction) | Monthly Newsletter | Appreciative | Loyalty tier upgrade, VIP early access to new categories |
| **Recently Complained** (any tenure) | Phone / Live Chat outreach | Empathetic | "A customer care agent will follow up to make sure everything is resolved" |

**Key principle:** Don't contact everyone the same way. Use the model score to **prioritize**
outreach and **personalize** the message. The top 20% highest-risk customers deserve immediate
personal attention. The next 30% can be handled via automated but personalized emails.
The rest can receive standard loyalty communications.

---
# Final Conclusions

## 1. Key EDA Findings

- The dataset is **imbalanced** (~17% churners) — accuracy alone is a misleading metric
- **Tenure** is the most important predictor: new customers churn more
- **Complaints** are a strong early warning signal for churn
- **Cashback programs** work as a retention incentive — churners receive lower cashback on average
- Satisfaction score alone is not a reliable churn predictor

## 2. Best Predictive Model

**Random Forest** outperforms Logistic Regression across all key metrics (Recall, F1, Lift).
For a real-world churn prediction system, Random Forest would be the recommended choice.
Logistic Regression remains valuable as a transparent fallback for regulatory or explainability purposes.

## 3. Main Churn Drivers

1. **Tenure** — short tenure = high churn risk
2. **Complaints** — unresolved complaints are a red flag
3. **Cashback Amount** — low perceived value leads to disengagement

## 4. Recommended Marketing Actions

- **Onboarding program** for customers in their first 90 days (free shipping, welcome bonus)
- **Complaint fast-response protocol** with a 24h SLA and automatic goodwill coupon
- **Personalized cashback booster** for at-risk customers identified by the model
- Use model probability scores to **tier** retention campaigns by urgency and cost

The most impactful insight: **proactive retention is cheaper than acquisition**.
If we can catch even 50% of churners before they leave with a targeted offer,
the ROI of this model more than justifies the investment in the retention program.